# Projeto Fictus | Analise de Vendas — Bloco 4: Riscos Ocultos

---

## Pergunta Central do Bloco
> **Quais riscos um comprador herdaria ao adquirir o ativo?**

---

## Contexto do Bloco

Com os limites operacionais mapeados no estágio anterior, a diligência agora se aprofunda no que as métricas agregadas podem esconder. Este bloco opera sob a lógica da investigação de vulnerabilidades, analisando pontos que costumam passar despercebidos em balanços financeiros tradicionais, como a volatilidade por categoria e a dependência de parceiros específicos.

Nota de Intenção: O objetivo aqui não é invalidar o negócio, mas sim calcular o "desconto de risco". Identificar riscos ocultos permite que o comprador negocie um preço justo e prepare planos de contenção para cenários de estresse logístico ou comercial.

**Este bloco investiga:**
1. Quais categorias apresentam maior volatilidade de receita?
2. A margem é excessivamente sensível a choques de frete?
3. Há dependências críticas de vendedores difíceis de substituir?
4. Onde estão os outliers estruturais que distorcem as médias?
5. O negócio depende estruturalmente de datas comemorativas?
6. Um dia a mais de lead time quanto impacta a satisfação do cliente?
7. Há concentração geográfica de receita que representa risco estrutural — se uma região desacelerar, quanto da receita total está em risco?
8. A taxa de cancelamento tem padrão identificável — concentra em categorias, regiões ou períodos específicos — ou é ruído distribuído?

---

## Nota Metodológica — Deslocamento Temporal
Dados originais Olist **2016–2018** deslocados **+7 anos** → período **2023–2025**.  
**Análise restrita a partir de janeiro/2024** — dados de 2023 desconsiderados.

---

## Configuração do Ambiente

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
from pathlib import Path

# ─── Caminhos relativos — funcionam em qualquer máquina ─────────────────────
# O notebook está em notebooks/ → a raiz é um nível acima
NOTEBOOK_DIR = Path().resolve()
# Detecta a raiz do projeto subindo a hierarquia de pastas
# Funciona em qualquer estrutura: raiz/, notebooks/, notebooks/vendas/
def _find_base(start: Path) -> Path:
    for p in [start, start.parent, start.parent.parent]:
        if (p / "data").exists() or (p / "notebooks").exists():
            return p
    return start
BASE_DIR = _find_base(NOTEBOOK_DIR)
DIR_PRE      = BASE_DIR / "data" / "pre-tratados"
DIR_EXT      = BASE_DIR / "data" / "externos"
DIR_EXPORTS  = BASE_DIR / "exports"
DIR_EXPORTS.mkdir(parents=True, exist_ok=True)


warnings.filterwarnings("ignore")

COR_RECEITA  = "#1B4F72"
COR_MARGEM   = "#27AE60"
COR_ALERTA   = "#C0392B"
COR_NEUTRO   = "#7F8C8D"
COR_DESTAQUE = "#E67E22"
COR_ROXO     = "#8E44AD"
sns.set_theme(style="whitegrid", font_scale=1.0)
plt.rcParams.update({
    "figure.dpi": 150, "savefig.dpi": 150, "savefig.bbox": "tight",
    "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})
def fmt_pct(x, pos=None): return f"{x:.1f}%"
def fmt_brl(x, pos=None):
    if abs(x) >= 1_000_000: return f"R$ {x/1_000_000:.1f}M"
    if abs(x) >= 1_000:     return f"R$ {x/1_000:.0f}K"
    return f"R$ {x:.0f}"
def salvar(fig, nome):
    caminho = DIR_EXPORTS / f"{nome}.png"
    fig.savefig(caminho)
    print(f"  → Salvo: {caminho.name}")
print("Ambiente configurado.")

## Carregamento dos Dados

In [ ]:
# ─── Separador padrão: vírgula, decimal ponto ────────────────────────────────
#   Todos os arquivos gerados pelo ETL usam sep=',' e decimal='.'
def ler_csv(caminho, sep=",", **kwargs):
    df = pd.read_csv(caminho, sep=sep, low_memory=False, **kwargs)
    df.columns = df.columns.str.strip()
    return df

fato  = ler_csv(DIR_PRE / "fato_vendas.csv")
dim_p = ler_csv(DIR_PRE / "dim_produto.csv")
dim_c = ler_csv(DIR_PRE / "dim_cliente.csv")
dim_v = ler_csv(DIR_PRE / "dim_vendedor.csv")
dim_t = ler_csv(DIR_PRE / "dim_tempo.csv")

# ─── Conversão de datas ───────────────────────────────────────────────────────
for col in ["data_compra", "data_entrega_cliente", "data_previsao_entrega",
            "data_envio_transportadora", "data_aprovacao"]:
    if col in fato.columns:
        fato[col] = pd.to_datetime(fato[col], errors="coerce")
dim_t["data"] = pd.to_datetime(dim_t["data"], errors="coerce")

# ─── Colunas numéricas: garantia adicional ────────────────────────────────────
for col in ["preco", "valor_frete", "valor_total_item", "valor_pagamento_total",
            "lead_time_dias", "atraso_dias", "nota_review", "numero_parcelas"]:
    if col in fato.columns:
        fato[col] = pd.to_numeric(fato[col], errors="coerce")

# ─── Enriquecimento e filtro ─────────────────────────────────────────────────
fato = fato.merge(dim_p[["id_produto", "nome_categoria_produto"]], on="id_produto",  how="left")
fato = fato.merge(dim_c[["id_cliente", "estado_cliente"]],         on="id_cliente",  how="left")
fato = fato.merge(dim_v[["id_vendedor", "estado_vendedor"]],        on="id_vendedor", how="left")
fato = fato.merge(
    dim_t[["id_data", "ano", "mes", "trimestre", "nome_mes", "ano_mes"]],
    on="id_data", how="left"
)
fato["periodo"] = fato["ano"].astype(str) + "-Q" + fato["trimestre"].astype(str)

DATA_INICIO = "2024-01-01"
fato = fato[fato["data_compra"] >= DATA_INICIO].copy()

fe  = fato[fato["status_pedido"] == "entregue"].copy()
fne = fato[fato["status_pedido"] != "entregue"].copy()
periodos_ord = sorted(fato["periodo"].dropna().unique())

print(f"[FILTRO] Dados a partir de: {DATA_INICIO}")
print(f"fato (filtrado)   : {len(fato):>8} linhas | {fato['data_compra'].min().date()} → {fato['data_compra'].max().date()}")
print(f"fato_entregues    : {len(fe):>8} linhas ({len(fe)/len(fato)*100:.1f}% do total)")
print(f"Categorias únicas : {fe['nome_categoria_produto'].nunique()}")
print(f"Produtos únicos   : {fe['id_produto'].nunique()}")
print(f"Trimestres        : {len(periodos_ord)} — {periodos_ord[0]} → {periodos_ord[-1]}")


---

## Análise 1 — Quais categorias apresentam maior volatilidade de receita?

> *"A estabilidade do faturamento é uma dimensão de risco distinta da sua relevância financeira. Através do quadrante de relevância versus volatilidade, identifica-se onde a incerteza de planejamento é crítica para a operação. Categorias que apresentam alto volume e alta oscilação representam um risco real ao fluxo de caixa, exigindo que o comprador valide se a estrutura de suprimentos suporta tais variações sem comprometer o capital de giro."*

**Framework:** Pareto + PDCA — análise de volatilidade estrutural  
**Entrega:** Quadrante relevância × volatilidade por categoria, identificando os riscos reais vs os riscos periféricos

**Como este script responde à pergunta:**
> Para separar risco real de risco periférico, o script cruza volatilidade com relevância. Calcula o desvio padrão do crescimento mensal de cada categoria e produz dois gráficos:
>
> 1. **Top 15 por volatilidade:** Lista as categorias que mais oscilam mês a mês com o percentual de receita de cada uma anotado. Uma categoria volátil com 0,5% da receita é risco periférico; a mesma volatilidade em uma categoria com 15% da receita é risco real.
> 2. **Quadrante relevância × volatilidade:** Cada ponto é uma categoria — eixo horizontal é o peso na receita, eixo vertical é a volatilidade. O quadrante superior direito é a zona de alto risco: categorias relevantes e instáveis simultaneamente. O tamanho do ponto indica há quantos meses a categoria está ativa — pontos maiores têm mais histórico e volatilidade mais confiável.

**Análise do Resultado:**
 Nem todo faturamento tem a mesma qualidade. Esta análise identifica quais áreas do negócio são "estáveis" e quais são "imprevisíveis". Categorias com alta volatilidade (sobe e desce constante) exigem muito mais esforço de gestão de estoque e caixa. Se o crescimento da empresa vem apenas de categorias voláteis, o risco de um mês ruim comprometer a operação é alto. O objetivo aqui é entender se o motor da empresa é constante ou se ela vive de "picos" difíceis de sustentar.

In [ ]:
rec_cat_mensal = (
    fe.groupby(["ano_mes", "nome_categoria_produto"])["preco"].sum()
    .reset_index().sort_values(["nome_categoria_produto", "ano_mes"])
)
rec_cat_mensal["mom"] = (
    rec_cat_mensal.groupby("nome_categoria_produto")["preco"]
    .pct_change(fill_method=None) * 100
)

vol_cat = (
    rec_cat_mensal.groupby("nome_categoria_produto")
    .agg(
        receita_total = ("preco", "sum"),
        receita_media = ("preco", "mean"),
        mom_std       = ("mom",   "std"),
        mom_medio     = ("mom",   "mean"),
        n_meses       = ("preco", "count"),
    )
    .reset_index()
)
vol_cat = vol_cat[vol_cat["n_meses"] >= 6].copy()
vol_cat["pct_receita_total"] = vol_cat["receita_total"] / vol_cat["receita_total"].sum() * 100

med_std = vol_cat["mom_std"].median()
med_rel = vol_cat["pct_receita_total"].median()
alto_risco = vol_cat[(vol_cat["mom_std"] > med_std) & (vol_cat["pct_receita_total"] > med_rel)]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Análise 1 — Volatilidade de Receita por Categoria", fontsize=13, fontweight="bold")

top_vol = vol_cat.nlargest(15, "mom_std").sort_values("mom_std", ascending=True)
cores_v = [COR_ALERTA if v > med_std else COR_MARGEM for v in top_vol["mom_std"]]
bars = axes[0].barh(
    [c.replace("_", " ")[:30] for c in top_vol["nome_categoria_produto"]],
    top_vol["mom_std"], color=cores_v, alpha=0.85
)
for bar, pct in zip(bars, top_vol["pct_receita_total"]):
    axes[0].text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
                 f"{pct:.1f}% receita", va="center", fontsize=7, color=COR_NEUTRO)
axes[0].set_xlabel("Desvio Padrão do Crescimento MoM (pp)")
axes[0].set_title("Top 15 Categorias por Volatilidade", fontsize=11)

sc = axes[1].scatter(
    vol_cat["pct_receita_total"], vol_cat["mom_std"],
    s=vol_cat["n_meses"] * 5 + 20,
    c=vol_cat["mom_medio"], cmap="RdYlGn", alpha=0.75,
    edgecolors="white", linewidth=0.5
)
plt.colorbar(sc, ax=axes[1], label="Crescimento MoM médio (%)")
axes[1].axhline(med_std, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)
axes[1].axvline(med_rel, color=COR_NEUTRO, linestyle=":", linewidth=1, alpha=0.5)
for _, row in vol_cat.nlargest(8, "receita_total").iterrows():
    axes[1].annotate(
        row["nome_categoria_produto"].replace("_", " ")[:20],
        (row["pct_receita_total"], row["mom_std"]),
        fontsize=7, xytext=(4, 3), textcoords="offset points"
    )
axes[1].text(med_rel * 1.1, vol_cat["mom_std"].max() * 0.92,
             "ALTO RISCO\n(relevante + volátil)", color=COR_ALERTA, fontsize=8, alpha=0.7)
axes[1].set_xlabel("% da Receita Total")
axes[1].set_ylabel("Volatilidade (std MoM, pp)")
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Quadrante: Relevância × Volatilidade", fontsize=11)

plt.tight_layout()
salvar(fig, "01_volatilidade_categorias")
plt.show()

print("\n" + "="*60)
print("INSIGHT — VOLATILIDADE POR CATEGORIA")
print("="*60)
print(f"Categorias de ALTO RISCO (relevante + volátil): {len(alto_risco)}")
for _, r in alto_risco.nlargest(5, "receita_total").iterrows():
    print(f"  {r['nome_categoria_produto']:<38} {r['pct_receita_total']:.1f}% receita | std MoM: {r['mom_std']:.1f}pp")

---

## Análise 2 — A margem é excessivamente sensível a choques de frete?

> *"A sensibilidade da margem a fatores externos de custo é uma dimensão crítica de risco estrutural. A simulação de choques progressivos de frete (+10%, +20% e +30%) quantifica diretamente o impacto sobre o resultado líquido por categoria — não sobre a demanda, mas sobre a viabilidade financeira de cada linha de produto. Categorias que se tornam inviáveis com choques moderados revelam onde a estrutura de precificação do ativo possui menor margem de segurança, orientando o comprador na priorização de renegociações comerciais antes do fechamento.""*

**Framework:** Matriz GUT — avaliação de gravidade e urgência do risco de frete  
**Entrega:** Simulação de choques de frete por categoria, com identificação de categorias que se tornam inviáveis e % da receita em risco

**Como este script responde à pergunta:**
> Em vez de apenas descrever o % de frete atual, o script simula três choques de aumento (+10%, +20%, +30%) e calcula para cada um quantas categorias ficam com resultado líquido negativo e qual percentual da receita isso representa.
>
> 1. **Impacto por categoria com choque +20%:** Barras horizontais mostram quanto cada categoria das 20 maiores perde em margem proxy. Barras vermelhas indicam queda acima de 15pp — categorias criticamente sensíveis. O Score GUT ao final traduz os resultados em linguagem de prioridade: gravidade, urgência e tendência.
> 2. **% frete atual × limiares de inviabilidade:** Plota o % de frete de cada categoria e traça linhas de limiar: categorias acima delas já estão na zona de risco com qualquer aumento de custo logístico.

**Análise do Resultado:**
Esta análise simula um cenário de estresse: 'E se o custo do transporte subir?'. Ela revela quais categorias ficam com resultado líquido negativo — receita não cobre o frete — se o custo logístico aumentar 20%. Para o comprador, isso identifica onde a estrutura financeira da operação tem menor margem de segurança e maior sensibilidade a variações externas de custo.Para o comprador, esse mapeamento orienta onde priorizar renegociação de condições comerciais ou revisão de precificação antes de fechar o negócio.

In [ ]:
base_cat = (
    fe.groupby("nome_categoria_produto")
    .agg(
        receita      = ("preco",       "sum"),
        frete_total  = ("valor_frete", "sum"),
        n_itens      = ("id_pedido",   "count"),
    )
    .reset_index()
)
base_cat["liquido_baseline"]   = base_cat["receita"] - base_cat["frete_total"]
base_cat["pct_frete_baseline"] = base_cat["frete_total"] / base_cat["receita"] * 100
base_cat["pct_receita"]        = base_cat["receita"] / base_cat["receita"].sum() * 100

CHOQUES = [0.10, 0.20, 0.30]
for choque in CHOQUES:
    label = f"choque_{int(choque*100)}pct"
    frete_chocado = base_cat["frete_total"] * (1 + choque)
    base_cat[f"liquido_{label}"]     = base_cat["receita"] - frete_chocado
    base_cat[f"impacto_{label}_pp"]  = (base_cat[f"liquido_{label}"] - base_cat["liquido_baseline"]) / base_cat["receita"] * 100
    base_cat[f"inviavel_{label}"]    = base_cat[f"liquido_{label}"] < 0

top20_sens = base_cat.nlargest(20, "receita").copy()

fig, axes = plt.subplots(1, 2, figsize=(16, 7))
fig.suptitle("Análise 2 — Sensibilidade da Margem a Choques de Frete (Matriz GUT)",
             fontsize=13, fontweight="bold")

sorted_sens = top20_sens.sort_values("impacto_choque_20pct_pp", ascending=True)
cores_imp = [COR_ALERTA if v < -15 else COR_DESTAQUE if v < -8 else COR_MARGEM
             for v in sorted_sens["impacto_choque_20pct_pp"]]
axes[0].barh(
    [c.replace("_", " ")[:28] for c in sorted_sens["nome_categoria_produto"]],
    sorted_sens["impacto_choque_20pct_pp"],
    color=cores_imp, alpha=0.85
)
axes[0].axvline(0, color="black", linewidth=0.8)
axes[0].set_xlabel("Impacto na Margem Líquida (pp)")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:+.1f}pp"))
axes[0].set_title("Impacto de Choque de Frete +20%\n(Top 20 por receita)", fontsize=11)

sorted_frete = top20_sens.sort_values("pct_frete_baseline", ascending=False)
x_pos = range(len(sorted_frete))
axes[1].bar(x_pos, sorted_frete["pct_frete_baseline"],
            color=[COR_ALERTA if v > 40 else COR_DESTAQUE if v > 25 else COR_MARGEM
                   for v in sorted_frete["pct_frete_baseline"]], alpha=0.85)
axes[1].axhline(100/1.3, color=COR_ALERTA, linestyle="--", linewidth=1.5,
                label=f"Limiar inviabilidade +30%: {100/1.3:.1f}%")
axes[1].axhline(100/1.2, color=COR_DESTAQUE, linestyle=":", linewidth=1.2,
                label=f"Limiar inviabilidade +20%: {100/1.2:.1f}%")
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(
    [c.replace("_", " ")[:15] for c in sorted_frete["nome_categoria_produto"]],
    rotation=45, ha="right", fontsize=7
)
axes[1].set_ylabel("% Frete sobre Receita")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("% Frete Atual × Limiares de Inviabilidade", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "02_sensibilidade_frete")
plt.show()

print("\n" + "="*60)
print("INSIGHT — SENSIBILIDADE A CHOQUES DE FRETE")
print("="*60)
for choque in CHOQUES:
    label = f"choque_{int(choque*100)}pct"
    n_inv = base_cat[f"inviavel_{label}"].sum()
    pct_r = base_cat[base_cat[f"inviavel_{label}"]== True]["pct_receita"].sum()
    print(f"  Choque +{int(choque*100)}%: {n_inv} categorias inviáveis → {pct_r:.1f}% da receita em risco")

# Score GUT automático
n_inv_20 = base_cat["inviavel_choque_20pct"].sum()
pct_rec_risco = base_cat[base_cat["inviavel_choque_20pct"]]["pct_receita"].sum()
gut_gravidade = 3 if pct_rec_risco > 10 else 2 if n_inv_20 > 0 else 1
gut_urgencia  = 3 if base_cat["pct_frete_baseline"].max() > 80 else 2 if base_cat["pct_frete_baseline"].max() > 50 else 1
gut_tendencia = 2
gut_score = gut_gravidade * gut_urgencia * gut_tendencia
print(f"\nScore GUT — Risco de Frete:")
print(f"  Gravidade  : {gut_gravidade}/3 | Urgência: {gut_urgencia}/3 | Tendência: {gut_tendencia}/3")
print(f"  Score total: {gut_score} ({'CRÍTICO' if gut_score >= 12 else 'ALTO' if gut_score >= 6 else 'MODERADO'})")

---

## Análise 3 — Há dependências críticas de vendedores difíceis de substituir?

> *"A dependência de parceiros comerciais (sellers) é um risco que pode se acentuar após mudanças no controle societário. A saída, migração ou renegociação de parceiros estratégicos pode impactar o faturamento de forma imediata. Mapear quais vendedores são críticos e quantificar o impacto financeiro de sua eventual saída permite classificar o nível de risco por cenário e definir planos de contingência para a preservação do volume transacional."*

**Framework:** Análise estrutural de dependências críticas — Pareto + PDCA  
**Entrega:** Mapa de sellers críticos com relevância, estabilidade e qualidade, e simulação do impacto da saída dos top sellers

**Como este script responde à pergunta:**
> O script classifica como "crítico" qualquer seller com mais de 1% da receita total e presença em mais de 50% dos trimestres — relevante e recorrente ao mesmo tempo.
>
> 1. **Scatter relevância × estabilidade dos top 50 sellers:** Cada ponto é um seller. O quadrante superior direito (vermelhos) são os críticos — se saírem, o impacto é imediato e difícil de substituir. O tamanho indica quantas categorias o seller atende.
> 2. **Simulação de saída:** Barras mostram quanto da receita seria perdida com a saída dos top 1, 3, 5 e 10 sellers. A linha tracejada vermelha marca o limiar crítico de 10% de perda.

**Análise do Resultado:**
 Aqui medimos o "risco de saída". Se 30% ou 40% das suas vendas dependem de apenas dois ou três vendedores (sellers), você não é dono do seu canal, você é refém dos seus parceiros. Se um desses grandes vendedores decidir sair da plataforma ou for para o concorrente, o faturamento desaba. O ideal é que o sucesso da empresa esteja distribuído em muitos parceiros pequenos e médios, tornando a operação mais resiliente e menos dependente de negociações individuais.

In [ ]:
pareto_v = (
    fe.groupby("id_vendedor")
    .agg(
        receita      = ("preco",                 "sum"),
        n_pedidos    = ("id_pedido",              "nunique"),
        n_categorias = ("nome_categoria_produto", "nunique"),
        nota_media   = ("nota_review",            "mean"),
        n_trimestres = ("periodo",                "nunique"),
    )
    .reset_index().sort_values("receita", ascending=False).reset_index(drop=True)
)
pareto_v["pct_receita"] = pareto_v["receita"] / pareto_v["receita"].sum() * 100
pareto_v["pct_acum"]    = pareto_v["pct_receita"].cumsum()
pareto_v["rank"]        = pareto_v.index + 1

n_trimestres_total = fe["periodo"].nunique()
pareto_v["pct_trim_ativo"] = pareto_v["n_trimestres"] / n_trimestres_total * 100
pareto_v["seller_critico"] = (
    (pareto_v["pct_receita"]   >= 1.0) &
    (pareto_v["pct_trim_ativo"] >= 50)
)

n_criticos = pareto_v["seller_critico"].sum()
pct_receita_criticos = pareto_v[pareto_v["seller_critico"]]["pct_receita"].sum()

# Simulação: impacto da saída dos top 1, 3, 5 sellers
cenarios_saida = []
for n in [1, 3, 5, 10]:
    pct_perdida = pareto_v.head(n)["pct_receita"].sum()
    cenarios_saida.append({"n_sellers": n, "pct_receita_perdida": pct_perdida})
df_cenarios = pd.DataFrame(cenarios_saida)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle("Análise 3 — Dependências Críticas de Vendedores", fontsize=13, fontweight="bold")

top50_plot = pareto_v.head(50).dropna(subset=["pct_trim_ativo"])
cores_crit = [COR_ALERTA if c else COR_RECEITA for c in top50_plot["seller_critico"]]
axes[0].scatter(
    top50_plot["pct_trim_ativo"], top50_plot["pct_receita"],
    c=cores_crit, s=top50_plot["n_categorias"] * 20 + 20,
    alpha=0.8, edgecolors="white", linewidth=0.5
)
axes[0].axhline(1.0, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5, label="Limiar relevância: 1%")
axes[0].axvline(50,  color=COR_NEUTRO, linestyle=":",  linewidth=1, alpha=0.5, label="Limiar estabilidade: 50%")
axes[0].text(65, axes[0].get_ylim()[1] * 0.85, f"CRÍTICOS: {n_criticos} sellers\n= {pct_receita_criticos:.1f}% da receita",
             color=COR_ALERTA, fontsize=9, ha="center", alpha=0.8)
axes[0].set_xlabel("% Trimestres Ativo")
axes[0].set_ylabel("% da Receita Total")
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title("Relevância × Estabilidade — Top 50 Sellers", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_ALERTA,  label="Seller crítico"),
    mpatches.Patch(color=COR_RECEITA, label="Demais"),
], frameon=False, fontsize=8)

cores_sim = [COR_DESTAQUE if v < 10 else COR_ALERTA for v in df_cenarios["pct_receita_perdida"]]
bars = axes[1].bar(
    [f"Saída\ntop {r['n_sellers']}" for _, r in df_cenarios.iterrows()],
    df_cenarios["pct_receita_perdida"],
    color=cores_sim, alpha=0.85
)
for bar, val in zip(bars, df_cenarios["pct_receita_perdida"]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f"{val:.1f}%", ha="center", fontsize=10, fontweight="bold")
axes[1].set_ylabel("% da Receita Perdida")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_title("Simulação: Impacto da Saída dos Top Sellers", fontsize=11)
axes[1].axhline(10, color=COR_ALERTA, linestyle="--", linewidth=1, alpha=0.6, label="Limiar crítico 10%")
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "03_dependencia_sellers")
plt.show()

print("\n" + "="*60)
print("INSIGHT — DEPENDÊNCIAS CRÍTICAS DE SELLERS")
print("="*60)
print(f"Sellers críticos             : {n_criticos} (>1% receita + >50% trimestres)")
print(f"Receita nos sellers críticos : {pct_receita_criticos:.1f}%")
for _, r in df_cenarios.iterrows():
    nivel = "CRÍTICO" if r["pct_receita_perdida"] >= 10 else "ALTO" if r["pct_receita_perdida"] >= 5 else "MODERADO"
    print(f"  Saída top {int(r['n_sellers'])} sellers: -{r['pct_receita_perdida']:.1f}% receita — {nivel}")

---

## Análise 4 — Onde estão os outliers estruturais que distorcem as médias?

> *"Pedidos que fogem da distribuição estatística normal podem fazer o negócio parecer mais rentável ou mais problemático do que a operação recorrente sustenta. A identificação de outliers severos em três dimensões simultâneas — preço, frete e lead time — garante que a avaliação do ativo reflita a performance estrutural da empresa, não eventos isolados. Pedidos anômalos em múltiplas dimensões ao mesmo tempo são os que mais distorcem médias agregadas e, quando concentrados em categorias específicas, indicam fragilidades operacionais que precisam ser endereçadas antes da transição de controle."*

**Framework:** Análise sistêmica de variabilidade — Controle Estatístico de Processo  
**Entrega:** Distribuição de preço, frete e lead time com identificação de outliers IQR k=3, e perfil dos pedidos anômalos em múltiplas dimensões

**Como este script responde à pergunta:**
> O script aplica o método IQR (intervalo interquartil) com fator k=3 para identificar outliers severos em três dimensões: preço, frete e lead time. Fator k=3 é conservador — identifica apenas casos genuinamente anômalos.
>
> 1. **Histogramas com limites de outlier:** Para cada dimensão, plota a distribuição e marca os limites do IQR k=3. O título de cada gráfico mostra quantos pedidos estão fora desses limites e qual % da receita representam.
> 2. **Pedidos anômalos em múltiplas dimensões:** Conta pedidos que são outliers em duas ou mais dimensões simultaneamente — esses distorcem as médias e podem levar a decisões equivocadas.

**Análise do Resultado:**
 Identificar esses outliers protege o comprador de ser enganado por médias: um custo médio de entrega pode parecer alto apenas por causa de 1% de pedidos com frete extremo, distorcendo a percepção da operação real. Saber onde estão esses casos anômalos é o primeiro passo para avaliar se representam exceções gerenciáveis ou fragilidades estruturais.

In [ ]:
def iqr_bounds(series, k=3.0):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

dims = {
    "preco":          {"label": "Preço (R$)",      "cor": COR_RECEITA},
    "valor_frete":    {"label": "Frete (R$)",       "cor": COR_DESTAQUE},
    "lead_time_dias": {"label": "Lead Time (dias)", "cor": COR_ROXO},
}

resumo_outliers = []
fe_out = fe.copy()
for col, cfg in dims.items():
    serie = fe[col].dropna()
    lo, hi = iqr_bounds(serie)
    mask = (fe[col] < lo) | (fe[col] > hi)
    n_out = mask.sum()
    pct_out = n_out / len(fe) * 100
    pct_rec = fe.loc[mask, "preco"].sum() / fe["preco"].sum() * 100 if col != "lead_time_dias" else None
    resumo_outliers.append({"col": col, "dimensao": cfg["label"], "n_outliers": n_out,
                            "pct": pct_out, "lower": lo, "upper": hi, "pct_receita": pct_rec})
    fe_out[f"flag_{col}"] = mask.fillna(False).astype(int)

fe_out["n_flags"] = fe_out[[f"flag_{c}" for c in dims.keys()]].sum(axis=1)
outliers_multiplos = fe_out[fe_out["n_flags"] >= 2][
    ["id_pedido", "preco", "valor_frete", "lead_time_dias", "nota_review",
     "nome_categoria_produto", "n_flags"]
]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle("Análise 4 — Outliers Estruturais (IQR k=3)", fontsize=13, fontweight="bold")

for i, (col, cfg) in enumerate(dims.items()):
    serie = fe[col].dropna()
    lo, hi = iqr_bounds(serie)
    serie_clean = serie[(serie >= serie.quantile(0.01)) & (serie <= serie.quantile(0.99))]
    axes[i].hist(serie_clean, bins=50, color=cfg["cor"], alpha=0.7, edgecolor="none")
    if lo > serie.min():
        axes[i].axvline(lo, color=COR_ALERTA, linewidth=1.5, linestyle="--", label=f"Inf: {lo:.1f}")
    axes[i].axvline(hi, color=COR_ALERTA, linewidth=1.5, linestyle="-", label=f"Sup: {hi:.1f}")
    r = resumo_outliers[i]
    rec_str = f" | {r['pct_receita']:.1f}% receita" if r["pct_receita"] is not None else ""
    axes[i].set_title(f"{cfg['label']}\n{r['n_outliers']} outliers ({r['pct']:.1f}%{rec_str})", fontsize=10)
    axes[i].set_xlabel(cfg["label"])
    axes[i].set_ylabel("Frequência")
    axes[i].legend(frameon=False, fontsize=7)

plt.tight_layout()
salvar(fig, "04_outliers_estruturais")
plt.show()

print("\n" + "="*60)
print("INSIGHT — OUTLIERS ESTRUTURAIS")
print("="*60)
for r in resumo_outliers:
    rec_str = f" | {r['pct_receita']:.1f}% da receita" if r["pct_receita"] is not None else ""
    print(f"  {r['dimensao']:<22} {r['n_outliers']:>5} outliers ({r['pct']:.1f}%){rec_str}")
print(f"\nPedidos anômalos em 2+ dimensões: {len(outliers_multiplos)}")
if len(outliers_multiplos) > 0:
    print(f"  Categorias mais afetadas:")
    for cat, cnt in outliers_multiplos["nome_categoria_produto"].value_counts().head(5).items():
        print(f"    {cat:<40} {cnt} pedidos")

---

## Análise 5 — O negócio depende estruturalmente de datas comemorativas?

> *"Um modelo de negócio cuja viabilidade depende excessivamente de picos sazonais (como Black Friday) apresenta maior risco de gestão de demanda. O cruzamento entre índices de sazonalidade e concentração de categorias determina se o resultado anual é estrutural ou se depende de janelas temporais curtas. Quantificar o percentual de receita em risco caso esses picos não se repitam é essencial para avaliar a sustentabilidade do fluxo de caixa ao longo do ciclo anual."*

**Framework:** PDCA — distinção entre crescimento estrutural e crescimento dependente de eventos  
**Entrega:** Índice de sazonalidade por mês, identificação dos meses críticos e % da receita anual concentrada em meses de pico

**Como este script responde à pergunta:**
> O script calcula o índice de sazonalidade de cada mês — receita média daquele mês dividida pela média geral. Um índice de 1,5 em novembro significa que novembro fatura 50% acima da média.
>
> 1. **Índice de sazonalidade por mês:** Barras em vermelho (pico acima de 1,3x), verde (acima da média) e cinza (abaixo). O limiar de 1,3x é o critério de "mês de pico". Se poucos meses concentram grande parte da receita, o negócio depende desses eventos para atingir as metas.
> 2. **Concentração da receita no ano:** Ordena os meses do maior para o menor e plota a curva acumulada. Quanto mais íngreme no início, mais concentrada é a receita em poucos meses.

**Análise do Resultado:**
 O negócio cresce de forma saudável ou depende de eventos pontuais para atingir suas metas? Uma operação com sazonalidade muito concentrada exige gestão cuidadosa de capital de giro — os meses de vale precisam ser financiados pelos picos. Para o comprador, entender esse ritmo é fundamental para avaliar a previsibilidade do fluxo de caixa e o risco de períodos de baixa demanda prolongados.

In [ ]:
MESES_ORD = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
             "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]

saz = (
    fe.groupby(["ano", "mes", "nome_mes"])
    .agg(receita=("preco", "sum"), n_pedidos=("id_pedido", "nunique"),
         nota_media=("nota_review", "mean"))
    .reset_index()
)
media_global = saz["receita"].mean()

indice_saz = (
    saz.groupby(["mes", "nome_mes"])
    .agg(receita_media=("receita", "mean"), pedidos_medio=("n_pedidos", "mean"))
    .reset_index().sort_values("mes")
)
indice_saz["indice_saz"] = indice_saz["receita_media"] / media_global
indice_saz["nome_mes_ord"] = pd.Categorical(indice_saz["nome_mes"], categories=MESES_ORD, ordered=True)
indice_saz = indice_saz.sort_values("nome_mes_ord")

# Meses de pico: índice > 1.3
meses_pico = indice_saz[indice_saz["indice_saz"] > 1.3]
pct_receita_pico = meses_pico["receita_media"].sum() / indice_saz["receita_media"].sum() * 100
pico = indice_saz.loc[indice_saz["indice_saz"].idxmax()]
vale = indice_saz.loc[indice_saz["indice_saz"].idxmin()]
ampl = pico["indice_saz"] / vale["indice_saz"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 5 — Dependência de Datas Comemorativas", fontsize=13, fontweight="bold")

x_m = range(len(indice_saz))
cores_saz = [COR_ALERTA if v > 1.3 else COR_MARGEM if v >= 1.0 else COR_NEUTRO
             for v in indice_saz["indice_saz"]]
bars = axes[0].bar(x_m, indice_saz["indice_saz"], color=cores_saz, alpha=0.85)
axes[0].axhline(1.0, color="black", linewidth=1)
axes[0].axhline(1.3, color=COR_ALERTA, linewidth=1, linestyle="--", alpha=0.6, label="Limiar pico: 1.3x")
for xi, val in zip(x_m, indice_saz["indice_saz"]):
    axes[0].text(xi, val + 0.01, f"{val:.2f}x", ha="center", va="bottom", fontsize=7)
axes[0].set_xticks(x_m)
axes[0].set_xticklabels([m[:3] for m in indice_saz["nome_mes"]], fontsize=9)
axes[0].set_ylabel("Índice (1.0 = média do período)")
axes[0].set_title("Índice de Sazonalidade de Receita", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_ALERTA, label=f"Pico (>1.3x) — {len(meses_pico)} meses"),
    mpatches.Patch(color=COR_MARGEM, label="Acima da média"),
    mpatches.Patch(color=COR_NEUTRO, label="Abaixo da média"),
], frameon=False, fontsize=8)

# % receita acumulada por mês
receita_por_mes = indice_saz["receita_media"].values
pct_por_mes = receita_por_mes / receita_por_mes.sum() * 100
pct_acum = np.cumsum(np.sort(pct_por_mes)[::-1])
axes[1].bar(range(len(pct_por_mes)), np.sort(pct_por_mes)[::-1],
            color=[COR_ALERTA if v > 10 else COR_RECEITA for v in np.sort(pct_por_mes)[::-1]], alpha=0.85)
ax_twin = axes[1].twinx()
ax_twin.plot(range(len(pct_acum)), pct_acum, color=COR_DESTAQUE, linewidth=2,
             marker="o", markersize=4, label="% acumulado")
ax_twin.axhline(50, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.5)
ax_twin.set_ylabel("% Receita Acumulada", color=COR_DESTAQUE)
ax_twin.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_ylabel("% da Receita Anual")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[1].set_xlabel("Meses (ordenados por receita)")
axes[1].set_title("Concentração da Receita ao Longo do Ano", fontsize=11)

plt.tight_layout()
salvar(fig, "05_sazonalidade_comemorativas")
plt.show()

print("\n" + "="*60)
print("INSIGHT — DEPENDÊNCIA DE DATAS COMEMORATIVAS")
print("="*60)
print(f"Mês de pico    : {pico['nome_mes']} ({pico['indice_saz']:.2f}x a média)")
print(f"Mês de vale    : {vale['nome_mes']} ({vale['indice_saz']:.2f}x a média)")
print(f"Amplitude      : {ampl:.1f}x entre pico e vale")
print(f"Meses de pico (>1.3x): {list(meses_pico['nome_mes'].values)} → {pct_receita_pico:.1f}% da receita anual")
nivel_dependencia = "ALTA" if ampl > 3.0 or pct_receita_pico > 30 else "MODERADA" if ampl > 2.0 else "BAIXA"
print(f"Dependência sazonal: {nivel_dependencia}")

---

## Análise 6 — Um dia a mais de lead time quanto impacta a satisfação do cliente?

> *"O risco operacional se traduz em risco de negócio quando compromete a experiência do consumidor e a taxa de recompra. A regressão entre lead time e nota de review quantifica exatamente o custo de cada dia adicional de atraso em pontos de satisfação — um coeficiente que permite calcular a perda potencial de receita futura com base no volume de pedidos atrasados. Estabelecer esse limiar define o SLA operacionalmente relevante: acima dele, a experiência do cliente se deteriora de forma mensurável e o risco de churn silencioso aumenta."*

**Framework:** Análise de causa e efeito — impacto operacional na satisfação  
**Entrega:** Correlação lead time × nota de review com regressão, distribuição de notas por faixa de prazo e estimativa do custo em satisfação por dia de atraso

**Como este script responde à pergunta:**
> O script usa regressão linear para quantificar exatamente quanto cada dia adicional de lead time reduz a nota de review — um coeficiente que traduz risco operacional em perda de satisfação mensurável.
>
> 1. **Scatter lead time × nota com regressão:** A linha de regressão mostra a direção e intensidade da relação. O coeficiente anotado diz quanto a nota cai por dia adicional — multiplicado por 7, mostra o impacto de uma semana a mais.
> 2. **Nota média por faixa de lead time:** Divide pedidos em seis faixas e calcula a nota média de cada uma. Mostra se a relação é linear ou se há um ponto de inflexão — o limiar de SLA operacionalmente relevante para a satisfação.

**Análise do Resultado:**
 Esta análise revela o "limite da paciência" do seu cliente. Mais do que um número, ela mostra se a satisfação do consumidor é sensível ao tempo de forma linear ou se existe um "penhasco": um dia específico onde a nota desaba. Se o gráfico mostrar que a nota cai drasticamente após um certo período, você descobriu o seu limite operacional real. Para qualquer gestor, esse dado é o que define o padrão de promessa de entrega; prometer menos do que isso é perder venda, entregar depois disso é destruir a reputação da marca.


In [ ]:
df_lt_nota = fe[["lead_time_dias", "nota_review", "entregue_no_prazo",
                  "atraso_dias", "nome_categoria_produto"]].dropna()

r_lt_nota, p_lt_nota = stats.pearsonr(df_lt_nota["lead_time_dias"], df_lt_nota["nota_review"])
m_lt, b_lt, _, _, _ = stats.linregress(df_lt_nota["lead_time_dias"], df_lt_nota["nota_review"])

# Nota média por faixa de lead time
bins   = [0, 7, 14, 21, 30, 60, 999]
labels = ["≤7d", "8-14d", "15-21d", "22-30d", "31-60d", ">60d"]
df_lt_nota["faixa_lt"] = pd.cut(df_lt_nota["lead_time_dias"], bins=bins, labels=labels)
nota_por_faixa = (
    df_lt_nota.groupby("faixa_lt", observed=True)
    .agg(nota_media=("nota_review", "mean"), n=("nota_review", "count"))
    .reset_index()
)

# Nota: no prazo vs atrasado
nota_no_prazo = df_lt_nota[df_lt_nota["entregue_no_prazo"] == 1]["nota_review"].mean()
nota_atrasado = df_lt_nota[df_lt_nota["entregue_no_prazo"] == 0]["nota_review"].mean()
delta_nota    = nota_atrasado - nota_no_prazo

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Análise 6 — Impacto do Lead Time na Satisfação do Cliente", fontsize=13, fontweight="bold")

# Scatter lead time × nota com regressão
sample = df_lt_nota.sample(min(3000, len(df_lt_nota)), random_state=42)
axes[0].scatter(sample["lead_time_dias"], sample["nota_review"],
                alpha=0.15, s=5, color=COR_RECEITA, rasterized=True)
xfit = np.linspace(df_lt_nota["lead_time_dias"].min(), df_lt_nota["lead_time_dias"].quantile(0.99), 100)
axes[0].plot(xfit, m_lt*xfit+b_lt, color=COR_ALERTA, linewidth=2, label=f"Regressão: {m_lt:.4f} pts/dia")
axes[0].axhline(df_lt_nota["nota_review"].mean(), color=COR_NEUTRO, linestyle="--",
                linewidth=1, alpha=0.7, label=f"Média global: {df_lt_nota['nota_review'].mean():.2f}")
axes[0].set_xlabel("Lead Time (dias)")
axes[0].set_ylabel("Nota de Review")
axes[0].set_ylim(0.5, 5.5)
axes[0].set_title(f"Lead Time × Nota de Review\ncorr={r_lt_nota:.2f} (p={p_lt_nota:.4f})", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)

# Nota por faixa de lead time
cores_faixa = [COR_MARGEM if v >= 4.0 else COR_DESTAQUE if v >= 3.5 else COR_ALERTA
               for v in nota_por_faixa["nota_media"]]
bars = axes[1].bar(range(len(nota_por_faixa)), nota_por_faixa["nota_media"],
                   color=cores_faixa, alpha=0.85)
for bar, (_, row) in zip(bars, nota_por_faixa.iterrows()):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{row['nota_media']:.2f}\n(n={row['n']:,})",
                 ha="center", va="bottom", fontsize=7)
axes[1].axhline(4.0, color=COR_NEUTRO, linestyle="--", linewidth=1, alpha=0.7, label="Benchmark 4.0")
axes[1].set_xticks(range(len(nota_por_faixa)))
axes[1].set_xticklabels(nota_por_faixa["faixa_lt"].astype(str), fontsize=9)
axes[1].set_ylabel("Nota Média de Review")
axes[1].set_ylim(0, 5.5)
axes[1].set_xlabel("Faixa de Lead Time")
axes[1].set_title("Nota Média por Faixa de Lead Time", fontsize=11)
axes[1].legend(frameon=False, fontsize=8)

plt.tight_layout()
salvar(fig, "06_lead_time_vs_nota")
plt.show()

print("\n" + "="*60)
print("INSIGHT — IMPACTO DO LEAD TIME NA SATISFAÇÃO")
print("="*60)
print(f"Correlação lead time × nota  : {r_lt_nota:.2f} (p={p_lt_nota:.4f})")
print(f"Regressão                    : {m_lt:.4f} pontos de nota por dia adicional")
print(f"  Ou seja: +7 dias de lead time = {m_lt*7:.2f} pontos a menos na nota")
print(f"Nota pedidos no prazo        : {nota_no_prazo:.2f}")
print(f"Nota pedidos atrasados       : {nota_atrasado:.2f}")
print(f"Impacto do atraso            : {delta_nota:.2f} pontos ({abs(delta_nota)/nota_no_prazo*100:.1f}% de perda)")

---

## Análise 7 — Há concentração geográfica de receita que representa risco estrutural?

> *"A concentração de faturamento em poucas regiões geográficas limita a escala e expõe o negócio a crises econômicas locais ou mudanças regulatórias específicas. Uma empresa com receita concentrada em poucos estados possui uma vulnerabilidade sistêmica maior do que uma operação com distribuição nacional equilibrada. O mapeamento geográfico permite calcular o raio de risco e a dependência da malha logística regional."*

**Framework:** Análise de concentração geoespacial + simulação de choque regional  
**Entrega:** Mapa de calor de receita por estado + simulação de impacto por desaceleração regional

**Como este script responde à pergunta:**
> A concentração geográfica de receita é um risco que não aparece nos resultados financeiros — só fica visível quando o problema já aconteceu. Este script torna esse risco explícito e quantificado com duas visualizações:
>
> 1. **Pareto de receita por estado:** Ordena os estados do maior para o menor e plota tanto o % individual quanto a curva acumulada. O ponto onde a curva cruza 80% revela quantos estados sustentam o negócio. Se 2 ou 3 estados concentram 80% da receita, o negócio é estruturalmente regional. A linha de corte dos 80% torna o nível de concentração imediatamente visual e comparável.
> 2. **Simulação de choque regional:** Para os top 5 estados, simula o que aconteceria com a receita total em três cenários de desaceleração: queda de 20%, 40% e 60% da receita daquele estado. O gráfico mostra a perda absoluta e percentual para cada estado em cada cenário. Estados com barras longas no cenário de -20% são os de maior risco sistêmico — uma desaceleração moderada já tem impacto relevante no resultado consolidado.

**Análise do Resultado:** Aqui medimos se o negócio é robusto ou se ele é dependente de uma "bolha" regional. Se a maior parte da receita vem de apenas um ou dois estados, a empresa está vulnerável. Um problema climático, uma greve de transportes local ou uma mudança de impostos naquela região pode paralisar o faturamento total. O teste de "choque" que o script faz mostra o tamanho do tombo: se uma queda de 20% no seu estado principal derruba sua receita global em um nível insustentável, o seu maior desafio não é vender mais, é diversificar onde você vende.


In [ ]:
# ─── Análise 7 — Concentração Geográfica de Receita ──────────────────────────

receita_estado = (
    fe.groupby("estado_cliente")
    .agg(
        receita_total = ("preco",      "sum"),
        n_pedidos     = ("id_pedido",  "nunique"),
        ticket_medio  = ("preco",      "mean"),
    )
    .reset_index()
    .sort_values("receita_total", ascending=False)
    .reset_index(drop=True)
)
total_geral = receita_estado["receita_total"].sum()
receita_estado["pct_receita"] = receita_estado["receita_total"] / total_geral * 100
receita_estado["pct_acum"]    = receita_estado["pct_receita"].cumsum()

n_estados_80 = int((receita_estado["pct_acum"] <= 80).sum()) + 1
top5_estados = receita_estado.head(5)

# Simulação de choque regional
choques = [0.20, 0.40, 0.60]
sim_data = []
for _, row in top5_estados.iterrows():
    for choque in choques:
        perda_abs = row["receita_total"] * choque
        perda_pct = perda_abs / total_geral * 100
        sim_data.append({
            "estado": row["estado_cliente"],
            "choque": f"-{int(choque*100)}%",
            "perda_pct_total": perda_pct,
        })
df_sim = pd.DataFrame(sim_data)

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Análise 7 — Concentração Geográfica de Receita: Risco Estrutural Regional", fontsize=13, fontweight="bold")

# Pareto de estados
x_e = range(len(receita_estado))
cores_estado = [COR_ALERTA if r["pct_acum"] <= 80 else COR_NEUTRO
                for _, r in receita_estado.iterrows()]
axes[0].bar(x_e, receita_estado["pct_receita"], color=cores_estado, alpha=0.85)
ax2_e = axes[0].twinx()
ax2_e.plot(x_e, receita_estado["pct_acum"], color=COR_DESTAQUE, linewidth=2, marker="o", markersize=3)
ax2_e.axhline(80, color="black", linestyle="--", linewidth=1, alpha=0.5)
ax2_e.set_ylabel("% acumulado", color=COR_DESTAQUE)
ax2_e.yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
ax2_e.tick_params(axis="y", labelcolor=COR_DESTAQUE)
axes[0].set_xticks(list(x_e))
axes[0].set_xticklabels(receita_estado["estado_cliente"].tolist(), rotation=45, ha="right", fontsize=8)
axes[0].set_ylabel("% da receita total")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))
axes[0].set_title(f"Pareto de Receita por Estado\n({n_estados_80} estados = 80% da receita | vermelho)", fontsize=11)
axes[0].legend(handles=[
    mpatches.Patch(color=COR_ALERTA, label=f"Top {n_estados_80} (80% da receita)"),
    mpatches.Patch(color=COR_NEUTRO, label="Demais estados"),
], frameon=False, fontsize=9)

# Simulação de choque
pivot_sim = df_sim.pivot(index="estado", columns="choque", values="perda_pct_total")
pivot_sim = pivot_sim.reindex(top5_estados["estado_cliente"].tolist())
x_s = range(len(pivot_sim))
width = 0.25
cores_choque = [COR_MARGEM, COR_DESTAQUE, COR_ALERTA]
for idx, (col, cor) in enumerate(zip(["-20%", "-40%", "-60%"], cores_choque)):
    offset = (idx - 1) * width
    bars = axes[1].bar(
        [xi + offset for xi in x_s],
        pivot_sim[col],
        width=width, color=cor, alpha=0.85, label=f"Choque {col} no estado"
    )
axes[1].set_xticks(list(x_s))
axes[1].set_xticklabels(pivot_sim.index.tolist(), fontsize=10)
axes[1].set_ylabel("Impacto na receita total (pp)")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"-{v:.1f}pp"))
axes[1].set_title("Simulação: Impacto de Desaceleração Regional\n(queda na receita total se aquele estado desacelerar)", fontsize=11)
axes[1].legend(frameon=False, fontsize=9)
axes[1].axhline(5, color=COR_ALERTA, linestyle="--", linewidth=1, alpha=0.6, label="Limiar crítico (5pp)")

plt.tight_layout()
salvar(fig, "07_concentracao_geografica")
plt.show()

top1_estado = receita_estado.iloc[0]
top1_impacto_20 = top1_estado["pct_receita"] * 0.20
print("\n" + "="*55)
print("INSIGHT — CONCENTRAÇÃO GEOGRÁFICA")
print("="*55)
print(f"Total de estados com receita       : {len(receita_estado)}")
print(f"Estados para 80% da receita        : {n_estados_80}")
print(f"Estado #1: {top1_estado['estado_cliente']:<5} — {top1_estado['pct_receita']:.1f}% da receita")
print(f"  Impacto de -20% neste estado     : -{top1_impacto_20:.1f}pp da receita total")
sinal_geo = "⚠️  RISCO ESTRUTURAL" if n_estados_80 <= 3 else "✅ DIVERSIFICADO" if n_estados_80 >= 8 else "⚠️  ATENÇÃO"
print(f"Nível de concentração geográfica   : {sinal_geo}")


---

## Análise 8 — A taxa de cancelamento tem padrão identificável ou é ruído distribuído?

> *"Cancelamentos recorrentes são indicadores de falhas sistêmicas em processos de venda ou logística. Distinguir cancelamentos aleatórios (ruído operacional) de padrões concentrados em categorias, regiões ou períodos específicos permite identificar causas raízes e calcular o custo de ineficiência do modelo. Padrões identificáveis sugerem gargalos de processo que precisam de intervenção estrutural imediata pós-aquisição."*

**Framework:** Análise de distribuição e concentração de falhas (Quality Management)  
**Entrega:** Decomposição da taxa de cancelamento por categoria, estado e período

**Como este script responde à pergunta:**
> A diferença entre ruído e padrão é o que determina se o cancelamento é um problema gerenciável ou um risco estrutural. Este script decompõe a taxa de cancelamento em três dimensões:
>
> 1. **Top categorias por taxa de cancelamento:** Calcula a taxa de cancelamento (pedidos cancelados / pedidos totais) de cada categoria e lista as 15 com maior taxa — com o volume de pedidos anotado ao lado para contextualizar se é um problema relevante ou marginal. Uma categoria com 30% de cancelamento e 5.000 pedidos é um problema crítico; a mesma taxa em 20 pedidos é estatisticamente irrelevante.
> 2. **Cancelamento por estado e por período:** Dois painéis mostram a variação geográfica e temporal da taxa de cancelamento. Se estados específicos aparecem consistentemente com taxa alta, há um problema logístico ou operacional regional. Se o cancelamento aumenta em períodos específicos, o problema é sazonal e possivelmente relacionado a capacidade de estoque ou de processamento nos picos de demanda.

**Análise do Resultado:**
 O cancelamento é o sinal mais honesto de que algo falhou. Esta análise separa o "ruído" (problemas que acontecem em qualquer lugar) de "padrões" (falhas repetitivas). Se os cancelamentos estão concentrados em categorias específicas ou regiões certas, você não tem um problema de cliente, você tem um problema de processo ou de fornecedor. Identificar esses focos permite agir cirurgicamente: às vezes, desativar um único parceiro ou categoria resolve 80% das reclamações e estornos da empresa.

In [ ]:
# ─── Análise 8 — Decomposição da Taxa de Cancelamento ────────────────────────

# Base completa (inclui cancelados)
cancelamentos = fato.copy()
cancelamentos["cancelado"] = (cancelamentos["status_pedido"] == "cancelado").astype(int)

# Por categoria
canc_cat = (
    cancelamentos.groupby("nome_categoria_produto")
    .agg(
        n_total    = ("id_pedido", "nunique"),
        n_cancelado= ("cancelado", "sum"),
    )
    .reset_index()
)
canc_cat["taxa_canc"] = canc_cat["n_cancelado"] / canc_cat["n_total"] * 100
canc_cat = canc_cat[canc_cat["n_total"] >= 30].sort_values("taxa_canc", ascending=False)
taxa_geral = cancelamentos["cancelado"].sum() / cancelamentos["id_pedido"].nunique() * 100

# Por estado
canc_estado = (
    cancelamentos.groupby("estado_cliente")
    .agg(n_total=("id_pedido", "nunique"), n_cancelado=("cancelado", "sum"))
    .reset_index()
)
canc_estado["taxa_canc"] = canc_estado["n_cancelado"] / canc_estado["n_total"] * 100
canc_estado = canc_estado[canc_estado["n_total"] >= 30].sort_values("taxa_canc", ascending=False)

# Por período (trimestre)
canc_trim = (
    cancelamentos.groupby("periodo")
    .agg(n_total=("id_pedido", "nunique"), n_cancelado=("cancelado", "sum"))
    .reset_index().sort_values("periodo")
)
canc_trim["taxa_canc"] = canc_trim["n_cancelado"] / canc_trim["n_total"] * 100

# ─── Plot ─────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("Análise 8 — Decomposição da Taxa de Cancelamento: Padrão ou Ruído?", fontsize=13, fontweight="bold")

# Por categoria (top 15)
top15_cat = canc_cat.head(15)
cores_cat = [COR_ALERTA if v > taxa_geral * 1.5 else COR_DESTAQUE if v > taxa_geral else COR_NEUTRO
             for v in top15_cat["taxa_canc"]]
bars = axes[0].barh(top15_cat["nome_categoria_produto"], top15_cat["taxa_canc"],
                    color=cores_cat, alpha=0.85)
axes[0].axvline(taxa_geral, color="black", linestyle="--", linewidth=1.2, label=f"Média geral: {taxa_geral:.1f}%")
for bar, n in zip(bars, top15_cat["n_total"]):
    axes[0].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f"{n:,} ped", va="center", fontsize=7)
axes[0].set_xlabel("Taxa de cancelamento (%)")
axes[0].set_title("Top 15 Categorias\npor Taxa de Cancelamento", fontsize=11)
axes[0].legend(frameon=False, fontsize=8)
axes[0].invert_yaxis()

# Por estado
cores_est = [COR_ALERTA if v > taxa_geral * 1.5 else COR_DESTAQUE if v > taxa_geral else COR_MARGEM
             for v in canc_estado["taxa_canc"]]
axes[1].barh(canc_estado["estado_cliente"], canc_estado["taxa_canc"], color=cores_est, alpha=0.85)
axes[1].axvline(taxa_geral, color="black", linestyle="--", linewidth=1.2)
axes[1].set_xlabel("Taxa de cancelamento (%)")
axes[1].set_title("Taxa de Cancelamento por Estado", fontsize=11)
axes[1].invert_yaxis()

# Por período
x_t = range(len(canc_trim))
cores_trim = [COR_ALERTA if v > taxa_geral * 1.5 else COR_DESTAQUE if v > taxa_geral else COR_NEUTRO
              for v in canc_trim["taxa_canc"]]
axes[2].bar(x_t, canc_trim["taxa_canc"], color=cores_trim, alpha=0.85)
axes[2].axhline(taxa_geral, color="black", linestyle="--", linewidth=1.2)
axes[2].set_xticks(list(x_t))
axes[2].set_xticklabels([str(p) for p in canc_trim["periodo"]], rotation=45, ha="right", fontsize=8)
axes[2].set_ylabel("Taxa de cancelamento (%)")
axes[2].set_title("Taxa de Cancelamento por Trimestre", fontsize=11)
axes[2].yaxis.set_major_formatter(mticker.FuncFormatter(fmt_pct))

plt.tight_layout()
salvar(fig, "08_decomposicao_cancelamentos")
plt.show()

# Métricas
cat_critica = canc_cat.iloc[0]
estado_critico = canc_estado.iloc[0]
variacao_temporal = canc_trim["taxa_canc"].max() - canc_trim["taxa_canc"].min()
n_cats_acima = (canc_cat["taxa_canc"] > taxa_geral * 1.5).sum()
n_est_acima  = (canc_estado["taxa_canc"] > taxa_geral * 1.5).sum()

print("\n" + "="*55)
print("INSIGHT — DECOMPOSIÇÃO DO CANCELAMENTO")
print("="*55)
print(f"Taxa de cancelamento geral          : {taxa_geral:.1f}%")
print(f"Categorias com taxa >1.5x a média   : {n_cats_acima}")
print(f"  Pior categoria: {cat_critica['nome_categoria_produto']} ({cat_critica['taxa_canc']:.1f}% | {cat_critica['n_total']:,} pedidos)")
print(f"Estados com taxa >1.5x a média      : {n_est_acima}")
print(f"  Pior estado: {estado_critico['estado_cliente']} ({estado_critico['taxa_canc']:.1f}%)")
print(f"Variação temporal (max-min)         : {variacao_temporal:.1f}pp")
tipo_canc = "padrão concentrado" if n_cats_acima >= 3 or n_est_acima >= 3 else "ruído distribuído"
print(f"Natureza dos cancelamentos          : {tipo_canc}")


---

## Síntese do Bloco 4 — Mapa de Riscos para o comprador
> **Limitações desta análise:** a simulação de choque de frete assume elasticidade de demanda zero — ou seja, que o cliente absorve o aumento sem reduzir compras. O impacto real pode ser maior. A análise de saída de sellers assume substituição zero a curto prazo, o que é conservador. A correlação lead time × satisfação é estatística — não implica que melhorar o lead time isoladamente resolverá o problema de nota sem considerar outros fatores (qualidade do produto, atendimento, experiência pós-venda).


In [ ]:

# Concentração geográfica (análise 7)
_rec_est_v = (
    fe.groupby("estado_cliente")["preco"].sum()
    .sort_values(ascending=False)
    .reset_index(name="receita")
)
_rec_est_v["pct"] = _rec_est_v["receita"] / _rec_est_v["receita"].sum() * 100
_rec_est_v["pct_acum"] = _rec_est_v["pct"].cumsum()
_n_est_80_v = int((_rec_est_v["pct_acum"] <= 80).sum()) + 1
s_geo = 3 if _n_est_80_v <= 3 else 2 if _n_est_80_v <= 6 else 1

# Cancelamentos (análise 8)
_canc_v = fato.copy()
_canc_v["cancelado"] = (_canc_v["status_pedido"] == "cancelado").astype(int)
_taxa_geral_v = _canc_v["cancelado"].sum() / _canc_v["id_pedido"].nunique() * 100
_canc_cat_v = (
    _canc_v.groupby("nome_categoria_produto")
    .agg(n_total=("id_pedido","nunique"), n_canc=("cancelado","sum"))
    .query("n_total >= 30")
    .assign(taxa=lambda d: d["n_canc"]/d["n_total"]*100)
)
_n_cats_criticas_v = int((_canc_cat_v["taxa"] > _taxa_geral_v * 1.5).sum())
s_cancelamento = 3 if _n_cats_criticas_v >= 5 else 2 if _n_cats_criticas_v >= 2 else 1

# Score automático por dimensão de risco
pct_alto_risco_cats = len(alto_risco) / len(vol_cat) * 100
s_volatilidade = 3 if pct_alto_risco_cats > 30 or med_std > 60 else 2 if pct_alto_risco_cats > 15 else 1
s_frete_gut    = 3 if gut_score >= 12 else 2 if gut_score >= 6 else 1
s_sellers_dep  = 3 if pct_receita_criticos > 30 else 2 if pct_receita_criticos > 15 else 1
s_outliers     = 3 if len(outliers_multiplos) > 50 else 2 if len(outliers_multiplos) > 10 else 1
s_sazonalidade = 3 if ampl > 3.0 or pct_receita_pico > 30 else 2 if ampl > 2.0 else 1
s_lead_nota    = 3 if abs(delta_nota) > 1.0 else 2 if abs(delta_nota) > 0.5 else 1

riscos = [
    ("Volatilidade de Receita",     s_volatilidade, f"{len(alto_risco)} cats relevantes e voláteis | std MoM médio: {med_std:.1f}pp"),
    ("Sensibilidade ao Frete",      s_frete_gut,    f"GUT score: {gut_score} | choque +20%: {pct_rec_risco:.1f}% receita em risco"),
    ("Dependência de Sellers",      s_sellers_dep,  f"{n_criticos} sellers críticos = {pct_receita_criticos:.1f}% da receita"),
    ("Outliers Estruturais",        s_outliers,     f"{len(outliers_multiplos)} pedidos anômalos em múltiplas dimensões"),
    ("Sazonalidade / Comemorativas",s_sazonalidade, f"amplitude {ampl:.1f}x | picos = {pct_receita_pico:.1f}% da receita anual"),
    ("Lead Time × Satisfação",      s_lead_nota,    f"atraso reduz nota em {abs(delta_nota):.2f} pts ({abs(delta_nota)/nota_no_prazo*100:.1f}%)"),
    ("Concentração Geográfica",      s_geo,          f"{_n_est_80_v} estados = 80% da receita"),
    ("Cancelamentos",                s_cancelamento, f"{_n_cats_criticas_v} categorias com taxa crítica | média geral: {_taxa_geral_v:.1f}%"),
]

fig, ax = plt.subplots(figsize=(12, 7))
ax.set_title("Bloco 4 — Mapa de Riscos para o comprador", fontsize=13, fontweight="bold")

nomes  = [r[0] for r in riscos]
scores = [r[1] for r in riscos]
descs  = [r[2] for r in riscos]
cores_risco = [COR_ALERTA if s == 3 else COR_DESTAQUE if s == 2 else COR_MARGEM for s in scores]

bars = ax.barh(nomes, scores, color=cores_risco, alpha=0.85, edgecolor="white")
ax.set_xlim(0, 3.8)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(["Baixo", "Médio", "Alto"], fontsize=10)
ax.axvline(1.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
ax.axvline(2.5, color=COR_NEUTRO, linewidth=0.5, alpha=0.4)
for bar, desc in zip(bars, descs):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            desc[:70] + ("..." if len(desc) > 70 else ""),
            va="center", fontsize=7.5, color="#333333")
ax.legend(handles=[
    mpatches.Patch(color=COR_MARGEM,   label="Baixo"),
    mpatches.Patch(color=COR_DESTAQUE, label="Médio"),
    mpatches.Patch(color=COR_ALERTA,   label="Alto"),
], frameon=False, fontsize=9, loc="lower right")

plt.tight_layout()
salvar(fig, "07_mapa_riscos")
plt.show()

NIVEL = {3: "🔴 ALTO", 2: "🟡 MÉDIO", 1: "🟢 BAIXO"}
print("\n" + "="*65)
print("SÍNTESE — BLOCO 4: MAPA DE RISCOS")
print("="*65)
for nome, score, desc in riscos:
    print(f"  {NIVEL[score]}  {nome:<30} {desc}")
print("="*65)
print("VEREDICTO PARCIAL DO BLOCO 4")
print("="*65)

---
*Próximo notebook: `05_EDA_cenarios_sobrevivencia.ipynb` — Sob quais condições o negócio sobrevive à aquisição?*

> Esta análise faz parte do **Projeto Fictus**, conduzido pela Lufi Data Consulting. Os três módulos analíticos — Vendas, Logística e Finanças — compõem a base do Relatório de Recomendação de Aquisição.